# Cert Prep — 04: UDFs & Spark SQL Functions

**Exam weight: ~10%**

Topics covered:
- Python UDF — registration, types, NULL handling
- Pandas UDF (vectorized) — Series→Series, grouped
- Built-in functions vs UDF performance
- UDF caveats: Gluten fallback, serialization overhead


In [1]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

Spark 4.0.2


Dataset ready
+---+-----+-----------+--------+----------+----------+
| id| name|       dept|  salary| hire_date|perf_score|
+---+-----+-----------+--------+----------+----------+
|  1|Alice|Engineering| 95000.0|2021-03-15|         4|
|  2|  Bob|  Marketing| 72000.0|2020-07-01|         3|
|  3|Carol|Engineering|105000.0|2019-11-20|         5|
|  4| Dave|  Marketing| 68000.0|2022-01-10|         2|
|  5|  Eve|Engineering| 88000.0|2021-09-05|         4|
|  6|Frank|         HR| 61000.0|2020-04-18|         3|
|  7|Grace|         HR|    NULL|2023-02-28|         1|
|  8| Hank|Engineering|112000.0|2018-06-12|         5|
|  9| Iris|  Marketing|    NULL|2022-11-03|         2|
| 10| Jack|         HR| 59000.0|2023-08-01|         1|
+---+-----+-----------+--------+----------+----------+



In [2]:
# ── Python UDF ───────────────────────────────────────────────────────────────
print("=== Python UDF ===")
from pyspark.sql.functions import udf

# Define and register
@udf(returnType=StringType())
def grade_salary(salary):
    if salary is None:
        return "Unknown"
    if salary >= 100000:
        return "A"
    elif salary >= 80000:
        return "B"
    elif salary >= 65000:
        return "C"
    else:
        return "D"

# Use in DataFrame
df.withColumn("grade", grade_salary(F.col("salary")))   .select("name","salary","grade").show()

# Register view + UDF for Spark SQL
df.createOrReplaceTempView("employees")
spark.udf.register("grade_salary_sql", grade_salary)
spark.sql("SELECT name, salary, grade_salary_sql(salary) as grade FROM employees").show()

=== Python UDF ===


+-----+--------+-------+
| name|  salary|  grade|
+-----+--------+-------+
|Alice| 95000.0|      B|
|  Bob| 72000.0|      C|
|Carol|105000.0|      A|
| Dave| 68000.0|      C|
|  Eve| 88000.0|      B|
|Frank| 61000.0|      D|
|Grace|    NULL|Unknown|
| Hank|112000.0|      A|
| Iris|    NULL|Unknown|
| Jack| 59000.0|      D|
+-----+--------+-------+

+-----+--------+-------+
| name|  salary|  grade|
+-----+--------+-------+
|Alice| 95000.0|      B|
|  Bob| 72000.0|      C|
|Carol|105000.0|      A|
| Dave| 68000.0|      C|
|  Eve| 88000.0|      B|
|Frank| 61000.0|      D|
|Grace|    NULL|Unknown|
| Hank|112000.0|      A|
| Iris|    NULL|Unknown|
| Jack| 59000.0|      D|
+-----+--------+-------+



In [3]:
# ── Pandas UDF (Vectorized) ───────────────────────────────────────────────────
print("=== Pandas UDF ===")
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Series → Series (element-wise, vectorized)
@pandas_udf(DoubleType())
def normalize_salary(salary_series: pd.Series) -> pd.Series:
    """Normalize salary to 0-1 range within the batch."""
    min_s = salary_series.min()
    max_s = salary_series.max()
    return (salary_series - min_s) / (max_s - min_s)

df.filter(F.col("salary").isNotNull())   .withColumn("norm_salary", normalize_salary(F.col("salary")))   .select("name","salary","norm_salary").show()

print()
print("Pandas UDF vs Python UDF:")
print("  Python UDF  — row-by-row (slow, serializes each row Python↔JVM)")
print("  Pandas UDF  — batch Arrow transfer (10-100x faster)")
print("  Built-in F. — JVM native, fastest — always prefer over UDF")

=== Pandas UDF ===


+-----+--------+-----------+
| name|  salary|norm_salary|
+-----+--------+-----------+
|Alice| 95000.0|        1.0|
|  Bob| 72000.0|        0.0|
|Carol|105000.0|        1.0|
| Dave| 68000.0|        0.0|
|  Eve| 88000.0|        1.0|
|Frank| 61000.0|        0.0|
| Hank|112000.0|        1.0|
| Jack| 59000.0|        0.0|
+-----+--------+-----------+


Pandas UDF vs Python UDF:
  Python UDF  — row-by-row (slow, serializes each row Python↔JVM)
  Pandas UDF  — batch Arrow transfer (10-100x faster)
  Built-in F. — JVM native, fastest — always prefer over UDF


In [4]:
# ── Built-in functions: always prefer over UDF ───────────────────────────────
print("=== Built-in Functions (no UDF needed) ===")

# These are all built-in — no UDF required
result = df.select(
    "name", "salary",
    # Math
    F.round(F.col("salary") / 1000, 1).alias("salary_k"),
    F.abs(F.col("salary") - 80000).alias("dist_from_80k"),
    F.pow(F.col("perf_score"), 2).alias("score_sq"),
    # String
    F.concat(F.col("name"), F.lit(" ("), F.col("dept"), F.lit(")")).alias("name_dept"),
    F.regexp_replace("dept", "ing", "").alias("dept_short"),
    # Conditional
    F.when(F.col("salary") > 90000, True).otherwise(False).alias("is_senior"),
    # Array / Map (for complex types)
    F.array(F.col("salary"), F.col("perf_score").cast("double")).alias("metrics"),
)
result.show(truncate=False)

=== Built-in Functions (no UDF needed) ===
+-----+--------+--------+-------------+--------+-------------------+----------+---------+---------------+
|name |salary  |salary_k|dist_from_80k|score_sq|name_dept          |dept_short|is_senior|metrics        |
+-----+--------+--------+-------------+--------+-------------------+----------+---------+---------------+
|Alice|95000.0 |95.0    |15000.0      |16.0    |Alice (Engineering)|Engineer  |true     |[95000.0, 4.0] |
|Bob  |72000.0 |72.0    |8000.0       |9.0     |Bob (Marketing)    |Market    |false    |[72000.0, 3.0] |
|Carol|105000.0|105.0   |25000.0      |25.0    |Carol (Engineering)|Engineer  |true     |[105000.0, 5.0]|
|Dave |68000.0 |68.0    |12000.0      |4.0     |Dave (Marketing)   |Market    |false    |[68000.0, 2.0] |
|Eve  |88000.0 |88.0    |8000.0       |16.0    |Eve (Engineering)  |Engineer  |false    |[88000.0, 4.0] |
|Frank|61000.0 |61.0    |19000.0      |9.0     |Frank (HR)         |HR        |false    |[61000.0, 3.0] |
|Gr